# 🤖 otto-GPT : 처음부터 만드는 나만의 한국어 언어모델

**100% 내 모델.** 토크나이저 · 모델 가중치 · 학습 전부 당신이 만듭니다.
가져다 쓰는 사전학습 모델은 **없습니다.**

| 항목 | 내용 |
|---|---|
| 아키텍처 | GPT (Decoder-only Transformer), nanoGPT 스타일 |
| 규모 | 약 **57M** 파라미터 (n_layer=10, n_embd=624) |
| 토크나이저 | SentencePiece BPE **직접 학습** (vocab 16K) |
| 환경 | Colab Pro (A100 권장, T4도 가능) |
| 핵심 기능 | **체크포인트 재개** — 데이터를 추가하며 계속 성장 |

### 성장 시나리오
```
1차 학습  →  체크포인트 저장 (Google Drive)
데이터 추가  →  체크포인트에서 이어서 학습  →  더 똑똑해진 모델
데이터 추가  →  이어서 학습  →  ...반복...
```
이것이 진짜 "키워나가는" 방식입니다. 추론 중 자동 학습이 아니라,
**데이터를 모을 때마다 이어서 사전학습**하는 통제된 성장입니다.

---
> ⚠️ 솔직한 기대치: 57M 모델은 GPT-4 같은 답을 하지 못합니다.
> 이 프로젝트의 가치는 **"언어모델의 전 과정을 내가 설계하고 통제했다"**는 증명입니다.
> 면접에서 "5.8B를 LoRA했다"보다 "토크나이저부터 GPT를 사전학습했다"가 훨씬 강력합니다.

## 0. 환경 설정 & GPU 확인

먼저 `런타임 → 런타임 유형 변경 → GPU`(가능하면 A100)를 선택하세요.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# 필요한 패키지 설치 (sentencepiece = 토크나이저 직접 학습용)
!pip install -q sentencepiece datasets
import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 1. Google Drive 연결 — 성장의 핵심

모델을 키워나가려면 체크포인트가 **세션이 끝나도 남아 있어야** 합니다.
Colab은 런타임이 끊기면 로컬 파일이 사라지므로, 모든 산출물(토크나이저·체크포인트·데이터)을
**Google Drive에 저장**합니다. 다음에 다시 와서 이어서 학습할 수 있습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 모든 산출물이 여기 저장됩니다. 다음 세션에서도 이 경로 그대로 쓰면 이어집니다.
PROJECT_DIR = '/content/drive/MyDrive/otto_gpt'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/ckpt', exist_ok=True)
print("프로젝트 폴더:", PROJECT_DIR)
print("내용:", os.listdir(PROJECT_DIR))

## 2. 학습 데이터 준비

언어모델은 데이터가 전부입니다. **이 데이터가 당신 모델의 정체성**입니다.

### 방법 A — 공개 한국어 데이터셋으로 시작 (권장)
HuggingFace의 한국어 위키피디아로 시작합니다. 처음엔 작게 받아서 파이프라인을
끝까지 돌려보고, 잘 되면 양을 늘리는 걸 권합니다.

### 방법 B — 내 데이터 추가
`{PROJECT_DIR}/data/` 안에 `.txt` 파일을 넣으면 자동으로 합쳐집니다.
나중에 의료 텍스트 등 도메인 데이터를 여기 넣어 모델을 특화시킬 수 있습니다.

In [ ]:
# 방법 A: 한국어 위키피디아 일부 다운로드
# streaming=True 로 받아서 원하는 양만큼만 저장 (전체는 수 GB)
from datasets import load_dataset

RAW_TXT = f'{PROJECT_DIR}/data/corpus_wiki.txt'

# 이미 받아둔 게 있으면 건너뛰기
if not os.path.exists(RAW_TXT):
    print("한국어 위키 다운로드 중... (처음엔 50,000 문서로 시작)")
    ds = load_dataset("wikimedia/wikipedia", "20231101.ko",
                      split="train", streaming=True)
    n_docs = 50_000   # ← 늘리고 싶으면 이 숫자를 키우세요
    with open(RAW_TXT, 'w', encoding='utf-8') as f:
        for i, doc in enumerate(ds):
            if i >= n_docs:
                break
            text = doc['text'].strip()
            if len(text) > 100:        # 너무 짧은 문서 제외
                f.write(text + '\n\n')
            if (i+1) % 10000 == 0:
                print(f"  {i+1} 문서 저장")
    print("완료:", RAW_TXT)
else:
    print("이미 존재:", RAW_TXT)

size_mb = os.path.getsize(RAW_TXT) / 1e6
print(f"코퍼스 크기: {size_mb:.1f} MB")

In [ ]:
# 모든 데이터 파일 합치기 (방법 A + 방법 B)
import glob

all_txt_files = glob.glob(f'{PROJECT_DIR}/data/*.txt')
print("발견된 데이터 파일:")
for p in all_txt_files:
    print(f"  {os.path.basename(p)}: {os.path.getsize(p)/1e6:.1f} MB")

# 하나로 합쳐 토크나이저/학습에 사용
MERGED_TXT = f'{PROJECT_DIR}/data/_merged.txt'
with open(MERGED_TXT, 'w', encoding='utf-8') as out:
    for p in all_txt_files:
        if os.path.basename(p) == '_merged.txt':
            continue
        with open(p, encoding='utf-8') as f:
            out.write(f.read() + '\n')
print(f"\n합본: {MERGED_TXT} ({os.path.getsize(MERGED_TXT)/1e6:.1f} MB)")

## 3. 토크나이저 직접 학습 (SentencePiece BPE)

**여기가 polyglot LoRA와 결정적으로 다른 지점입니다.**
남의 토크나이저를 가져오지 않고, 당신 데이터로 어휘를 직접 만듭니다.

한국어는 교착어라 BPE(Byte-Pair Encoding)가 잘 맞습니다.
- vocab_size=16000: 한국어 형태소를 적절히 담는 크기
- `character_coverage=0.9995`: 한글/한자/기호를 폭넓게 커버

In [ ]:
import sentencepiece as spm

SPM_PREFIX = f'{PROJECT_DIR}/tokenizer_ko'
VOCAB_SIZE = 16000

if not os.path.exists(SPM_PREFIX + '.model'):
    print("토크나이저 학습 중... (몇 분 소요)")
    spm.SentencePieceTrainer.train(
        input=MERGED_TXT,
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type='bpe',
        character_coverage=0.9995,
        # 특수 토큰: 0=pad, 1=unk(unk_id), bos/eos 지정
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        user_defined_symbols=['<sep>', '<pad2>'],
        input_sentence_size=1_000_000,   # 너무 크면 샘플링
        shuffle_input_sentence=True,
    )
    print("완료:", SPM_PREFIX + '.model')
else:
    print("이미 존재:", SPM_PREFIX + '.model')

# 로드 & 테스트
sp = spm.SentencePieceProcessor(model_file=SPM_PREFIX + '.model')
print("\n어휘 크기:", sp.get_piece_size())
sample = "인공지능은 사람처럼 학습하는 기술이다."
ids = sp.encode(sample)
print(f"\n원문: {sample}")
print(f"토큰: {[sp.id_to_piece(i) for i in ids]}")
print(f"ID:   {ids}")
print(f"복원: {sp.decode(ids)}")

## 4. 코퍼스를 토큰 배열로 변환

전체 텍스트를 토큰 ID로 바꿔 `.bin` 파일(uint16 배열)로 저장합니다.
nanoGPT 방식 — 학습 중 디스크에서 빠르게 랜덤 접근하기 위함입니다.
이 작업도 한 번 해두면 Drive에 남아 재사용됩니다.

In [ ]:
import numpy as np

TRAIN_BIN = f'{PROJECT_DIR}/data/train.bin'
VAL_BIN = f'{PROJECT_DIR}/data/val.bin'

if not (os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN)):
    print("코퍼스 토크나이징 중...")
    # 줄 단위로 인코딩하며 eos 추가
    all_ids = []
    with open(MERGED_TXT, encoding='utf-8') as f:
        buf = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            ids = sp.encode(line)
            buf.extend(ids)
            buf.append(sp.eos_id())
            if len(buf) > 1_000_000:
                all_ids.append(np.array(buf, dtype=np.uint16))
                buf = []
        if buf:
            all_ids.append(np.array(buf, dtype=np.uint16))
    data = np.concatenate(all_ids)
    print(f"총 토큰 수: {len(data):,}")

    # 90% train / 10% val
    n = len(data)
    split = int(n * 0.9)
    data[:split].tofile(TRAIN_BIN)
    data[split:].tofile(VAL_BIN)
    print(f"train: {split:,} 토큰 → {TRAIN_BIN}")
    print(f"val:   {n-split:,} 토큰 → {VAL_BIN}")
else:
    print("이미 존재. 토큰 수 확인:")
    print(f"  train: {os.path.getsize(TRAIN_BIN)//2:,}")
    print(f"  val:   {os.path.getsize(VAL_BIN)//2:,}")

## 5. GPT 모델 정의 (직접 구현)

GPT 아키텍처를 처음부터 구현합니다. 핵심 구성요소:
- **Causal Self-Attention**: 미래 토큰을 못 보게 마스킹 (PyTorch의 `scaled_dot_product_attention` = FlashAttention 사용)
- **Weight tying**: 입력 임베딩과 출력 head 가중치 공유 (파라미터 절약)
- **Pre-LayerNorm**: 안정적 학습

이 코드는 가져온 게 아니라 당신이 이해하고 통제하는 모델입니다.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 16000
    block_size: int = 512     # 한 번에 보는 최대 토큰 길이 (context)
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 624
    dropout: float = 0.1
    bias: bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3*cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.n_head, self.n_embd, self.dropout = cfg.n_head, cfg.n_embd, cfg.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        # FlashAttention (causal=True 가 미래 마스킹 처리)
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4*cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4*cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # 잔차 연결
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),   # 토큰 임베딩
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),   # 위치 임베딩
            drop=nn.Dropout(cfg.dropout),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   # weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        return n - self.transformer.wpe.weight.numel()

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-1)
            return logits, loss
        logits = self.lm_head(x[:, [-1], :])
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("GPT 모델 정의 완료")

## 6. 하이퍼파라미터 설정

Colab GPU에 맞춰 배치 크기를 조절합니다.
- **A100 (40GB)**: batch_size 24~32 가능
- **T4 (16GB)**: batch_size 8 + gradient accumulation 권장

`gradient_accumulation_steps`로 작은 배치를 모아 큰 배치 효과를 냅니다
(T4에서도 큰 유효 배치로 안정적 학습 가능).

In [ ]:
# ===== 학습 설정 (자유롭게 조정) =====
config = GPTConfig(
    vocab_size=VOCAB_SIZE,
    block_size=512,
    n_layer=10,
    n_head=12,
    n_embd=624,
    dropout=0.1,
)

# 배치 설정 — GPU에 맞게 자동 추천
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 38:        # A100 40GB
        batch_size = 24
        grad_accum = 2
    elif vram_gb >= 22:      # L4 / A10 등
        batch_size = 12
        grad_accum = 4
    else:                    # T4 16GB
        batch_size = 6
        grad_accum = 8
else:
    batch_size, grad_accum = 2, 1

# 유효 배치 = batch_size * grad_accum * block_size 토큰
learning_rate = 3e-4
min_lr = 3e-5
warmup_iters = 200
max_iters = 5000          # ← 한 세션에서 돌릴 step 수. 늘리면 더 학습.
eval_interval = 250       # 검증 + 체크포인트 저장 주기
eval_iters = 50
weight_decay = 0.1
grad_clip = 1.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'

print(f"디바이스: {device}, dtype: {dtype}")
print(f"batch_size={batch_size}, grad_accum={grad_accum}")
print(f"유효 배치: {batch_size*grad_accum*config.block_size:,} 토큰/step")

m_tmp = GPT(config)
print(f"모델 파라미터: {m_tmp.num_params()/1e6:.1f}M")
del m_tmp

## 7. 데이터 로더

`.bin`에서 랜덤하게 `block_size` 길이의 조각을 뽑아 배치를 만듭니다.
`x`는 입력, `y`는 한 칸 밀린 정답(다음 토큰 예측).

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode='r')
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - config.block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+config.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+config.block_size].astype(np.int64)) for i in ix])
    if device == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

# 테스트
xb, yb = get_batch('train')
print("배치 shape:", xb.shape, yb.shape)
print("샘플 디코드:", sp.decode(xb[0][:30].tolist()))

## 8. 학습 루프 — 체크포인트 재개 지원 ⭐

**이 셀이 프로젝트의 핵심입니다.**

- 시작할 때 Drive에 체크포인트가 있으면 **자동으로 이어서** 학습합니다.
- `iter_num`(누적 step), optimizer 상태, best loss까지 모두 복원합니다.
- `eval_interval`마다 검증 후, 최고 성능이면 체크포인트를 Drive에 저장합니다.

세션이 끊겨도, 다음에 이 노트북을 다시 열고 셀을 순서대로 실행하면
**멈춘 지점부터 계속 성장**합니다.

In [ ]:
import time

CKPT_PATH = f'{PROJECT_DIR}/ckpt/otto_gpt.pt'

# ---- 모델 & 옵티마이저 생성 ----
model = GPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=(0.9, 0.95), weight_decay=weight_decay)
scaler = torch.amp.GradScaler(enabled=(dtype == 'float16'))

# ---- 체크포인트 있으면 이어서 ----
iter_num = 0
best_val_loss = float('inf')
if os.path.exists(CKPT_PATH):
    print(f"✓ 체크포인트 발견 → 이어서 학습: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    # config 일치 확인 (모델 크기 안 바뀌었는지)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    iter_num = ckpt['iter_num']
    best_val_loss = ckpt['best_val_loss']
    print(f"  복원: iter_num={iter_num:,}, best_val_loss={best_val_loss:.4f}")
else:
    print("새로 학습 시작 (체크포인트 없음)")

print(f"학습 파라미터: {model.num_params()/1e6:.1f}M\n")

# ---- 학습률 스케줄 (warmup + cosine decay) ----
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it > max_iters:
        return min_lr
    ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)) if device=='cuda' else torch.no_grad():
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def save_checkpoint(val_loss):
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'config': config.__dict__,
        'iter_num': iter_num,
        'best_val_loss': val_loss,
    }, CKPT_PATH)
    print(f"    💾 체크포인트 저장 (val_loss={val_loss:.4f})")

# ---- 학습 루프 ----
print("학습 시작...\n")
model.train()
t0 = time.time()
target_iter = iter_num + max_iters    # 이번 세션에서 추가로 돌릴 만큼

while iter_num < target_iter:
    lr = get_lr(iter_num)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # gradient accumulation
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for micro in range(grad_accum):
        X, Y = get_batch('train')
        if device == 'cuda':
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)):
                _, loss = model(X, Y)
                loss = loss / grad_accum
        else:
            _, loss = model(X, Y)
            loss = loss / grad_accum
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    # 로그
    if iter_num % 50 == 0:
        dt = time.time() - t0
        print(f"iter {iter_num:5d} | loss {accum_loss:.4f} | lr {lr:.2e} | {dt:.1f}s")
        t0 = time.time()

    # 검증 + 저장
    if iter_num % eval_interval == 0 and iter_num > 0:
        losses = estimate_loss()
        print(f"  >> eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            save_checkpoint(best_val_loss)

    iter_num += 1

# 마지막 저장
final_losses = estimate_loss()
print(f"\n학습 종료. 최종 val_loss: {final_losses['val']:.4f}")
if final_losses['val'] < best_val_loss:
    best_val_loss = final_losses['val']
save_checkpoint(best_val_loss)
print(f"누적 학습 step: {iter_num:,}")

## 9. 생성 테스트 — 내 모델로 문장 만들기

학습한 모델로 직접 텍스트를 생성합니다.
초반(step 적을 때)엔 어색하지만, 학습을 거듭할수록 좋아집니다.

In [ ]:
def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    model.eval()
    ids = sp.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        if device == 'cuda':
            with torch.amp.autocast(device_type='cuda', dtype=getattr(torch, dtype)):
                out = model.generate(x, max_new_tokens, temperature, top_k)
        else:
            out = model.generate(x, max_new_tokens, temperature, top_k)
    return sp.decode(out[0].tolist())

for prompt in ["인공지능은", "한국의 역사는", "오늘 날씨가"]:
    print(f"[입력] {prompt}")
    print(f"[생성] {generate_text(prompt, max_new_tokens=80)}\n")

## 10. 🌱 모델을 계속 키우는 방법

이 노트북의 진짜 목적. 두 가지 성장 방식이 있습니다.

### 방식 1 — 같은 모델을 더 오래 학습 (가장 간단)
1. 이 노트북을 다시 엽니다.
2. 셀을 **0번부터 순서대로** 실행합니다.
3. 8번 셀이 자동으로 체크포인트를 찾아 **이어서** 학습합니다.
4. `max_iters`만큼 더 돌고 다시 저장됩니다.

→ 데이터는 그대로, 학습량만 늘려 모델을 다듬는 방식.

### 방식 2 — 새 데이터를 추가하고 학습 (지식 확장)
1. `{PROJECT_DIR}/data/` 에 새 `.txt` 파일을 넣습니다 (예: 의료 텍스트).
2. **2번 셀의 데이터 합치기**를 다시 실행합니다.
3. ⚠️ **토크나이저는 그대로 둡니다** (3번 셀 건너뛰기). 어휘를 바꾸면 기존 체크포인트와 호환 안 됨.
4. **4번 셀**을 다시 실행해 `train.bin`을 새로 만듭니다. (기존 .bin 삭제 후)
5. 8번 셀에서 이어서 학습 → 새 지식이 모델에 들어갑니다.

→ 도메인 데이터를 점점 추가해 모델을 특화/확장하는 방식.

### 더 큰 모델로 키우고 싶다면 (고급)
n_layer/n_embd를 키우면 **체크포인트가 호환되지 않습니다**(크기가 다름).
이땐 새 모델을 처음부터 학습하거나, 작은 모델 가중치를 큰 모델에 이식하는
**depth up-scaling** 기법이 필요합니다 — 마침 Kanana 리포트에 나온 그 기법입니다.
필요하면 그 단계도 도와드릴 수 있습니다.

---

### 면접/포트폴리오용 정리 포인트
이 프로젝트를 설명할 때 강조할 것:
1. **토크나이저부터 직접** — SentencePiece BPE를 내 데이터로 학습
2. **아키텍처 직접 구현** — Causal attention, weight tying, pre-LN 등 이해하고 작성
3. **재개 가능한 학습 파이프라인** — 체크포인트 관리, 누적 step 추적
4. **정직한 규모 인식** — 57M 모델의 한계를 알고, 무엇을 증명하려는지 명확
5. **확장 로드맵** — 데이터 추가, depth up-scaling까지 설계

In [ ]:
# 현재 모델 상태 요약
print("="*50)
print("otto-GPT 현재 상태")
print("="*50)
print(f"파라미터:      {model.num_params()/1e6:.1f}M")
print(f"누적 학습 step: {iter_num:,}")
print(f"최고 val_loss:  {best_val_loss:.4f}")
print(f"어휘 크기:      {sp.get_piece_size():,}")
print(f"체크포인트:     {CKPT_PATH}")
print(f"토크나이저:     {SPM_PREFIX}.model")
print("="*50)
print("\n다음에 이어서 학습하려면: 이 노트북을 다시 열고 셀을 순서대로 실행하세요.")